# 04 Threat hunting analysis

## Purpose
Re-derive the two hunts interactively so the reasoning can be inspected and
re-run, rather than presented only as finished prose. The hunt reports record the
conclusion; this notebook shows the queries behind it.

## Inputs
- `data/processed/DS001.jsonl`, `DS002.jsonl`, digest-verified.
- The recorded hunt reports `hunts/HUNT-001.md` and `hunts/HUNT-002.md`.

## Methodology
HUNT-002 tests whether the outbound web connection is command and control. The
decisive query is the connection count and the capture's resolved-time span, not
the port. HUNT-001 tests whether cross-host service execution can be attributed to
a second workstation, which requires separating records that can order events from
records that can only attribute them.

## Outputs
Printed query results. The durable artefacts are the hunt reports themselves,
which record the conclusion, the negative results and the method boundary.

## Conclusion
Written at the end of the notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name != "Cyberion-ThreatShield":
    ROOT = Path(r"{root}")
VENV = (ROOT / ".venv/Scripts/python.exe").resolve()
if Path(sys.executable).resolve() != VENV:
    raise RuntimeError(
        "Select the Cyberion (.venv) kernel. This analysis must run on the pinned "
        f"environment, not {sys.executable}."
    )
sys.path.insert(0, str(ROOT / "scripts"))

import test_all as harness

In [ ]:
frames = harness.load_datasets(ROOT)

# HUNT-002, T1071.001: how many connections does the suspect process make, and
# over what period? A channel is repetition over time, so the span is the finding.
ds2 = frames["DS002"]
suspect = "{d273d0f0-8078-5f67-cc06-000000000800}"
net = ds2[(ds2["event_id"].isin([3, 22])) & (ds2["raw"].map(
    lambda record: record.get("ProcessGuid") == suspect))]
print(f"connections and DNS by the suspect process: {len(net)}")
for _, row in net.iterrows():
    record = row["raw"]
    print(f"  line {row['record_locator']} eid={row['event_id']} "
          f"dest={record.get('DestinationIp')}:{record.get('DestinationPort')} "
          f"query={record.get('QueryName')!r} utc={row['timestamp_utc']}")

resolved = [t for t in ds2["timestamp_utc"] if t]
print(f"\nresolved-time span: {min(resolved)} -> {max(resolved)}")

In [ ]:
# Is port 80 a usable discriminator in this environment? Count every port-80
# destination and the application that opened it.
firewall = ds2[ds2["event_id"] == 5156]
port80 = firewall[firewall["raw"].map(lambda record: record.get("DestPort") == "80")]
print(f"5156 records: {len(firewall)}, of which port 80: {len(port80)}")
tally = {}
for record in port80["raw"]:
    key = (record.get("DestAddress"), record.get("Application", "?").split("\\")[-1])
    tally[key] = tally.get(key, 0) + 1
for (address, application), count in sorted(tally.items(), key=lambda item: -item[1]):
    print(f"  {address:<16} {application:<26} {count}")

In [ ]:
# HUNT-001: which cross-host RPC flows exist, and can each one be ordered?
rpc = ds2[ds2["event_id"] == 5156]
rpc = rpc[rpc["raw"].map(lambda record: record.get("DestPort") in ("135", "49726"))]
print("cross-host or DCOM RPC flows")
for _, row in rpc.iterrows():
    record = row["raw"]
    ordered = "ordered" if row["timestamp_utc"] else "ATTRIBUTED ONLY, no resolved time"
    print(f"  line {row['record_locator']:>5} on {row['host']:<26} "
          f"{record.get('SourceAddress')}:{record.get('SourcePort')} -> "
          f"{record.get('DestAddress')}:{record.get('DestPort')} "
          f"app={record.get('Application', '').split(chr(92))[-1]:<16} {ordered}")

## Conclusion

HUNT-002 stands at Inconclusive, and the queries above are why. The suspect
process made exactly one connection. The resolved-time span is a little over two
minutes, which cannot contain enough intervals to characterise a beacon. The
destination is inside RFC 1918 space, so "external command and control" cannot be
asserted. The port-80 tally is the most useful single result: the overwhelming
majority of port-80 flows in this capture are cloud platform traffic, which is why
T1071.001 has no rule and why writing one from this data would have been wrong.

HUNT-001 stands at Partially supported. The service execution is confirmed by a
ProcessGuid join inside one recording: the Service Control Manager reached a
dynamic RPC port and 0.578 seconds later became the recorded parent of a command
shell. The attribution to a second workstation is not established, because the
records naming the remote application carry no resolved event time and can
therefore attribute a flow but not order it. Reporting the pair as one ordered
sequence would have required inventing a timezone, which the timestamp policy
forbids.

Both conclusions are negative or partial and neither was forced positive.